<h1> Policy Iteration Algorithm

Objective:

To find the optimal policy:

$\pi^*$

without experiencing the environment.

In other words, unlike the Multi-Armed Bandit problem—where the agent learns through trial and error—here, since we possess a model of the environment:

$Transition Model
+
Reward Function$

we can determine the best policy through computation.

In [29]:
import gymnasium as gym
import numpy as np

from src.policy_iteration import (
    policy_evaluation, 
    policy_improvement,
    policy_iteration,
)

<h3> Step 1: Build The Environment 

In [30]:
env = gym.make(
    "FrozenLake-v1",
    is_slippery=False
)

S F F F</br>
F H F H</br>
F F F H</br>
H F F G

<h3> Step 2: Parameters

In [31]:
gamma = 0.99

n_states = env.observation_space.n

n_actions = env.action_space.n

<h3> Step 3: Random Policy

In [32]:
policy = np.ones(
    (
        n_states,
        n_actions
    )
) / n_actions

Each State: 
- Left   25%
- Down   25%
- Right  25%
- Up     25%

<h3> Step 4: Value Function

Initially, we have no information about the values ​​of the states.

In [33]:
V = np.zeros(
    n_states
)

<h3> Step 5: Policy Evaluation

In [34]:
V = policy_evaluation(
    env,
    n_states,
    policy,
    V,
    gamma=0.99,
    theta=1e-8,    
)

print(V)

[0.01235611 0.01042444 0.01933842 0.00947774 0.01478704 0.
 0.03889445 0.         0.03260247 0.08433764 0.13781085 0.
 0.         0.17034482 0.43357944 0.        ]


This is our first value function for the random policy.

When the Agent enters the Hole, the Episode ends, because there is no longer any future. So For example:

$V(5)=0$

This value function corresponds to a random policy.
In other words, the agent isn't intelligent yet. So, the values ​​simply indicate the worth of each state if we continue following this same random policy—
—not what the best path is.

#### Policy Evaluation Result

The initial policy was evaluated using the Bellman Expectation Equation.

The resulting value function represents the expected return of each state under a random policy.

Terminal states such as holes have zero value because the episode ends immediately after entering them.

The goal state also has zero value because the reward is received when transitioning into the goal, not while staying in the terminal state.

States closer to the goal have higher values because they have a higher probability of reaching the final reward under the current policy.

<h3> Step 6: Policy Improvement

So far:
- We had a random policy.
- We calculated the value of each state using policy evaluation.

Now, the question:
If we choose the best action in each state, does the policy improve?

Idea:

For each state:

We try all actions.
We calculate the value of each action:

$Q(s,a)$

which we call the action-value function.

Formula:

$Q(s,a)=
\sum_{s'}
P(s'|s,a)
[
R+\gamma V(s')
]$

Meaning:

If, in the current state,
we choose action $a$,
what amount of reward do we expect to receive?

In [35]:
new_policy = policy_improvement(
    env,
    n_states,
    n_actions,
    V,
    gamma=0.99
)

In [36]:
print(new_policy)

[[0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]]


In [37]:
action_map = {
    0:"←",
    1:"↓",
    2:"→",
    3:"↑"
}

for state in range(n_states):

    action = np.argmax(
        new_policy[state]
    )

    print(
        state,
        action_map[action]
    )

0 ↓
1 →
2 ↓
3 ←
4 ↓
5 ←
6 ↓
7 ←
8 →
9 ↓
10 ↓
11 ←
12 ←
13 →
14 →
15 ←


Action:

State 15 → Left

This is never used, because no episode continues after reaching the goal.

<h3> Step 7: Policy Iteration Algorithm

Policy Stability = Stopping Criterion:

If the new policy is identical to the previous policy, it means there is no further improvement.

Thus, Old Policy == New Policy, and the algorithm terminates.

In [38]:
optimal_policy, optimal_V = policy_iteration(
    env,
    n_states,
    n_actions,
    gamma=0.99
)

In [39]:
print(
    optimal_V.reshape(4,4)
)

[[0.95099005 0.96059601 0.970299   0.96059601]
 [0.96059601 0.         0.9801     0.        ]
 [0.970299   0.9801     0.99       0.        ]
 [0.         0.99       1.         0.        ]]


In [40]:
action_map = {
    0:"←",
    1:"↓",
    2:"→",
    3:"↑"
}

for state in range(n_states):
    action = np.argmax(
        optimal_policy[state]
    )

    print(
        state,
        action_map[action]
    )

0 ↓
1 →
2 ↓
3 ←
4 ↓
5 ←
6 ↓
7 ←
8 →
9 ↓
10 ↓
11 ←
12 ←
13 →
14 →
15 ←


<h3> Step 8: Policy Evaluation

In [41]:
def run_policy_episode(
    env,
    policy
):
    state, info = env.reset()
    total_reward = 0
    steps = 0

    while True:
        action = np.argmax(
            policy[state]
        )

        state, reward, terminated, truncated, info = env.step(action)

        total_reward += reward
        steps += 1

        if terminated or truncated:
            break

    return total_reward, steps

In [42]:
reward, steps = run_policy_episode(
    env,
    optimal_policy
)

print(
    reward,
    steps
)

1 6


This is the first instance where the agent:

- Does not learn from random experience.
- Has no neural network.
- Uses no gradients.

It arrives at the optimal policy solely through:

$A reward function
+
The Bellman equation
$

This is precisely the power of dynamic programming in classical RL.

### Comparison with Random Agent

Before: Random Policy

Success rate ≈ 1.5%

After: Policy Iteration

Success = 100%

And the path: 6 steps

This difference demonstrates the impact of Dynamic Programming.

### Policy Iteration Results

Policy Iteration successfully improved the initial random policy into an optimal policy by repeatedly applying policy evaluation and policy improvement.

The final value function shows higher values for states closer to the goal, while terminal states maintain zero value.

The learned policy avoids holes and reaches the goal efficiently.

The final agent achieved a successful episode with:

- Reward: 1
- Steps: 6

Compared to the random baseline (~1.5% success rate), Policy Iteration demonstrates a significant improvement by using the known transition model and Bellman equations.